# JiT-S2-VMamba — ConvGLU FFN (TransNeXt)

The FFN of every block is the token-wise SwiGLU: each token is processed on its own, so **all** local 2-D mixing happens inside SS2D, whose four 1-D scans reach a token's vertical neighbours only `W` steps later. This notebook swaps it for TransNeXt's **Convolutional GLU** (Shi, CVPR 2024), with its GELU replaced by our SiLU:

$$[g,\ v] = W_1 u + b_1,\qquad y = W_2\big(\mathrm{SiLU}(\mathrm{DWConv}_{3\times3}(g)) \odot v\big) + b_2$$

It is exactly SwiGLU with a depthwise 3×3 conv over the 8×8 token grid on the **gate** half, before its SiLU: each token's gate is computed from its 3×3 neighbourhood (TransNeXt's "gating by nearest-neighbour features"), while the value branch stays token-wise.

| FFN | per block (D=384) | model |
|---|---|---|
| SwiGLU (baseline) | 1,182,080 | 31.03M |
| **ConvGLU (this one)** | 1,192,320 (+10,240) | **31.15M** |
| GLUMBConv (sister notebook) | 1,202,176 (+20,096) | 31.27M |

Width is param-matched to SwiGLU (`h = int(4·384·2/3) = 1024`, the same 2/3 rule TransNeXt applies). DWConv init follows TransNeXt: `N(0, √(2/9))`, zero bias. Every conditioning arm is **off** (adaLN-Zero baseline scaffold), so the only variable vs the baseline is the FFN.

---

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The conv FFNs (src/primitives.py: ConvGLUFFN, GLUMBConvFFN),
# the ffn flag in vmamba.py, the build_model wiring in BOTH run_experiment.py
# and evaluate.py, the ffn configs, and the tests are all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the ffn wiring is present in both entry points
!grep -q '"ffn"' scripts/run_experiment.py && grep -q '"ffn"' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the FFN  *(first session only; ~1-2 min)*

In [ ]:
# === Verify the FFN (~1-2 min; first session only -- comment out on resume).
# 1) CPU proofs (reference scan, no GPU): ffn=swiglu bit-identical to HEAD for
#    10 existing arms; ConvGLU / GLUMBConv equal verbatim transcriptions of the
#    TransNeXt / SANA modules (float64); identity-DW ConvGLU == SwiGLU; prefix
#    tokens see only the centre tap (zero prefix<->grid Jacobian); param deltas;
#    train/eval build_model agree on both ffn configs.
# 2) GPU check: cuDNN depthwise conv vs a CPU float64 copy, both configs (and
#    +prefix) forward+backward under bf16 autocast, and the it/s vs swiglu --
#    note the it/s for the complexity table.
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If either FAILS, STOP -- do not train on an FFN that is not the one reported.
!PYTHONPATH=. python tests/test_ffn_cpu.py
!PYTHONPATH=. python scripts/check_ffn_cuda.py

## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
FFN        = "convglu"   # this notebook: TransNeXt ConvGLU (DWConv on the gate)
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = f"configs/tiny_imagenet/jit-s2-vmamba-ffn-{FFN}.yaml"
OUT_DIR = f"/kaggle/working/exp_ffn_{FFN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). Every conditioning arm stays OFF (adaLN-Zero baseline scaffold), so
# the FFN is the only variable vs the baseline.
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s = re.sub(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "ffn": FFN, "in_context_len": 0,
                 "state_init": "none", "ssc": "none", "adaln_cond": "full"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_ffn_convglu".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "| ffn:", FFN, "| arms: all off (adaLN-Zero baseline)")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 6. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + ffn now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
RUN = f"exp_ffn_{FFN}"
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the baseline and every other arm, so the only variable is the FFN.
- Sister notebook: `jit-s2-vmamba-tinyin-ffn-glumbconv.ipynb` (SANA Mix-FFN). The pair separates **where** the conv sits: gate only (ConvGLU) vs value + gate after a SiLU expansion (GLUMBConv).
- +0.12M params (+0.4% of the model): near param-matched, so a gap is architectural, not capacity. The depthwise conv's FLOPs are negligible, but it is an extra memory-bound kernel -- record the it/s printed by the verify cell for the complexity table.
- In-context prefix tokens (if ever combined with an in-context arm) are not on the grid: they see only the conv's centre tap, so the FFN never mixes prefix and image tokens (`tests/test_ffn_cpu.py` check D).
- `ffn` is orthogonal to the conditioning arms: to combine the winning FFN with the best arm, add `ffn: convglu` to that arm's config -- no code change.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.